# Rebuild derived/v1 on Databricks Free Edition

This notebook rebuilds the team dataset `derived/v1` (the tables the app and `referral_saathi.ipynb` read) from
the provided Marketplace data, with the repository's own rules-only scripts. Nothing is fitted by hand here and no
algorithm is changed: the notebook only exports the data, runs the scripts in their `run_all.sh` order and checks
the result against `derived/v1/MANIFEST.json`.

1. Export `facilities`, `india_post_pincode_directory` and `nfhs_5_district_health_indicators` from the
   Marketplace catalog to Parquet in a Unity Catalog volume, in the same layout, key order and column types as the
   local snapshot `data/raw/`.
2. Copy the scripts to local disk and install the pinned packages they were built with.
3. Run the audit steps the cleaning reads, then `cleaning/geo_identity`, `cleaning/claims` and
   `pipelines/source_scoring`, one script at a time.
4. Write the rebuilt `derived/v1` to the volume.
5. Compare row counts and hashes with `derived/v1/MANIFEST.json`, table by table.

How to run: import the repository into the workspace (a Git folder, or
`databricks workspace import-dir <repo> /Workspace/Users/<you>/databricksdatathon`), open this notebook, attach
serverless compute and Run all. About 10 minutes. The volume in `out_root` must exist (default: volume `uploads`
in schema `workspace.referral_saathi`); the notebook writes only under `out_root`.

What does not run here, and why:
- The Goal 3 page check (`s3_pagecheck.py`) fetches about 400 live web pages. It is not re-run: its stored result
  `derived/v1/source_page_checks.parquet` is reused when the repository or the fallback folder has it. Without it
  the scores are built with no page attribution (`FETCH=0` in `run_all.sh`), which moves about 6,100 of 518,191
  data points, and the check below reports those tables as different.
- Hand labels, gold labels and web verdicts are team files, not Marketplace data. The private repository has them.
  A code-only public copy leaves some out (see `tools/public_exclude.txt`); a step whose input is missing is
  skipped with a printed note, and the steps that need its output are skipped in turn. To include them, upload the
  files to `<out_root>/fallback_inputs/` under their repository paths (for example
  `fallback_inputs/audit/identity/labels/strong_groups_hand.csv`).
- The website crawl, the knowledge base (`kb/`), the external official lists and the model (`model/`) are not part
  of `derived/v1` and are not rebuilt here.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

dbutils.widgets.text("catalog", "databricks_virtue_foundation_dataset_dais_2026")
dbutils.widgets.text("schema", "virtue_foundation_dataset")
dbutils.widgets.text("out_root", "/Volumes/workspace/referral_saathi/uploads/rebuild")
dbutils.widgets.text("repo_root", "")  # empty: the folder above this notebook

CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
OUT_ROOT = Path(dbutils.widgets.get("out_root"))
REPO = Path(dbutils.widgets.get("repo_root") or Path.cwd().parent)
WORK = Path("/tmp/datathon_rebuild/repo")  # local disk: the scripts write intermediates next to themselves
DEPS = Path("/tmp/datathon_rebuild/deps")
FALLBACK = OUT_ROOT / "fallback_inputs"
assert (REPO / "pipelines/source_scoring/run_all.sh").exists(), f"repository not found at {REPO}; set repo_root"
print("repository:", REPO)
print("output root:", OUT_ROOT)

## 1. Export the Marketplace tables

Same query as `data/snapshot_scripts/export.py` (`SELECT * ORDER BY key`), same Arrow types (string, double,
int64), one Parquet file per table under `<out_root>/data/raw/`.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

TABLES = {"facilities": ["unique_id"], "india_post_pincode_directory": ["pincode", "officename"],
          "nfhs_5_district_health_indicators": ["state_ut", "district_name"]}
ARROW = {"string": pa.string(), "double": pa.float64(), "bigint": pa.int64()}
RAW = OUT_ROOT / "data/raw"
RAW.mkdir(parents=True, exist_ok=True)
for name, keys in TABLES.items():
    sdf = spark.table(f"`{CATALOG}`.`{SCHEMA}`.`{name}`").orderBy(*keys)
    schema = pa.schema([(f.name, ARROW[f.dataType.simpleString()]) for f in sdf.schema.fields])
    table = pa.Table.from_pandas(sdf.toPandas(), schema=schema, preserve_index=False).replace_schema_metadata(None)
    tmp = Path("/tmp") / f"{name}.parquet"
    pq.write_table(table, tmp)
    shutil.copyfile(tmp, RAW / tmp.name)
    print(f"{name}: {table.num_rows:,} rows, {table.num_columns} columns -> {RAW / tmp.name}")

## 2. Working copy and packages

The scripts run from a copy on local disk with `DATATHON_ROOT` (the copy) and `DATATHON_DATA_ROOT`
(`<out_root>/data`). Every Parquet file in the repository is left out of the copy, so every table is rebuilt from
the export, except the stored page check. The packages are the versions `derived/v1` was built with, installed
into a separate folder so the notebook's own Spark environment is untouched.

In [ ]:
PINS = ["pandas==3.1.1", "numpy==2.5.5", "pyarrow==26.0.1", "duckdb==2.0.1", "rapidfuzz==3.14.6"]
if sys.version_info < (3, 12):  # numpy 2.5 needs Python 3.12 (serverless environment 3 or later)
    PINS[1] = "numpy<2.5"
    print("Python", sys.version.split()[0], "- numpy pin relaxed to numpy<2.5; float columns may differ in the last digits")
if not (DEPS / "pandas").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                    "--target", str(DEPS), *PINS], check=True)
PY = sys.executable
DEPS_ENV = {"PYTHONPATH": str(DEPS)}
check = "import sys, pandas, numpy, pyarrow, duckdb, rapidfuzz; print(sys.version.split()[0], pandas.__version__, numpy.__version__, pyarrow.__version__, duckdb.__version__, rapidfuzz.__version__)"
print("python, pandas, numpy, pyarrow, duckdb, rapidfuzz:",
      subprocess.run([PY, "-c", check], env={**os.environ, **DEPS_ENV}, capture_output=True, text=True, check=True).stdout)

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
skip = shutil.ignore_patterns("__pycache__", "*.parquet", ".ipynb_checkpoints")
for d in ("audit", "cleaning", "pipelines", "model/gold"):
    shutil.copytree(REPO / d, WORK / d, ignore=skip)
(WORK / "derived/v1").mkdir(parents=True)
stored = REPO / "derived/v1/source_page_checks.parquet"
if stored.exists():
    shutil.copy2(stored, WORK / "derived/v1" / stored.name)
overlay = sorted(p for p in FALLBACK.rglob("*") if p.is_file()) if FALLBACK.exists() else []
for p in overlay:
    dst = WORK / p.relative_to(FALLBACK)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dst)
print(f"working copy: {sum(1 for p in WORK.rglob('*') if p.is_file()):,} files; {len(overlay)} from {FALLBACK}")

## 3. Run the pipeline

Order: the audit tables the cleaning reads (PIN centroids, facility geo, merge signal, scale flags, sentence
flags, audit grades), then the steps of `cleaning/geo_identity/run_all.sh`, `cleaning/claims/run_all.sh` and
`pipelines/source_scoring/run_all.sh`. A step whose input file is missing is skipped with a note; any other error
stops the notebook at the end of this cell.

In [ ]:
STEPS = [  # (directory, command)
    ("audit/geo", "pin_centroids.py"), ("audit/geo", "facility_geo.py"), ("audit/geo", "merge_location_signal.py"),
    ("audit/scale", "build_flags.py"), ("audit/merge", "detect.py"), ("audit/merge", "analyze.py"),
    ("audit/claims", "audit.py"),
    *[("cleaning/geo_identity", f"{s}.py") for s in ("s1_pin_gazetteer", "s2_facility_location", "s3_resolve",
                                                     "s4_groups", "s5_trust", "s6_eval", "s7_summary", "places")],
    *[("cleaning/claims", s) for s in ("lexicon.py", "build_claims.py", "grade.py", "evaluate.py", "measure.py",
                                       "handcheck.py score")],
    *[("pipelines/source_scoring", s) for s in ("ss_common.py", "package.py --hashes-before", "s1_sources.py",
                                                "s2_gold.py", "s4_datapoints.py", "s5_fit.py", "s6_score.py",
                                                "s7_eval.py", "package.py", "selfcheck.py")],
]
MISSING = ("FileNotFoundError", "No such file or directory", "No files found that match")
ENV = {**os.environ, **DEPS_ENV, "DATATHON_ROOT": str(WORK), "DATATHON_DATA_ROOT": str(OUT_ROOT / "data"),
       "PYTHONDONTWRITEBYTECODE": "1"}
LOGS = WORK / "_logs"
LOGS.mkdir(exist_ok=True)


def run_step(cwd: str, cmd: str) -> str:
    """Run one script; return ok, skipped (an input file is missing) or failed."""
    if cmd == "s4_datapoints.py":  # s3_pagecheck.py would run here; it fetches live pages, so it never runs
        found = (WORK / "derived/v1/source_page_checks.parquet").exists()
        print("     s3_pagecheck.py: " + ("reusing the stored source_page_checks.parquet" if found else
              "SKIPPED, no stored source_page_checks.parquet: scores are built without page attribution"))
    t0 = time.time()
    r = subprocess.run([PY, *cmd.split()], cwd=WORK / cwd, env=ENV, capture_output=True, text=True)
    (LOGS / f"{cwd.replace('/', '_')}_{cmd.split()[0]}.log").write_text(r.stdout + "\n--- stderr ---\n" + r.stderr)
    last = (r.stderr.strip().splitlines() or [""])[-1]
    status = "ok" if r.returncode == 0 else "skipped" if any(m in r.stderr for m in MISSING) else "failed"
    print(f"{status:>7}  {cwd}/{cmd}  ({time.time() - t0:.0f} s)" + ("" if status == "ok" else f"\n         {last}"))
    if status == "failed":
        print(r.stderr[-3000:])
    return status


results = {f"{cwd}/{cmd}": run_step(cwd, cmd) for cwd, cmd in STEPS}
print({s: list(results.values()).count(s) for s in ("ok", "skipped", "failed")})
skipped = [k for k, v in results.items() if v == "skipped"]
if skipped:
    print("Skipped because an input is not in this copy of the repository (see the note above each one):",
          *skipped, sep="\n  ")
assert "failed" not in results.values(), "a step failed for a reason other than a missing input; see above"

## 4. Write derived/v1 to the volume

`package.py` copies the canonical cleaned tables into `derived/v1` and writes `MANIFEST.json`. If it was skipped,
the cleaned tables that were rebuilt are copied here by the same mapping, so they are still published.

In [ ]:
COPY = json.loads(subprocess.run([PY, "-c", "import json; from package import COPY; print(json.dumps(COPY))"],
                                 cwd=WORK / "pipelines/source_scoring", env=ENV, capture_output=True, text=True,
                                 check=True).stdout)
DST = OUT_ROOT / "derived/v1"
shutil.rmtree(DST, ignore_errors=True)
DST.mkdir(parents=True)
built = WORK / "derived/v1"
for p in sorted(built.iterdir()):
    if p.is_file():
        shutil.copyfile(p, DST / p.name)
for name, src in COPY.items():
    if not (built / name).exists() and (WORK / src).exists():
        shutil.copyfile(WORK / src, DST / name)
        print("package.py did not run; copied", src)
print(f"{len(list(DST.iterdir()))} files in {DST}")

## 5. Check against MANIFEST.json

For each table in the committed `derived/v1/MANIFEST.json`: the row count, and the SHA-256 where one is known (the
nine cleaned tables have theirs in `other_input_sha256`; in the private repository every committed table can be
hashed). Where bytes differ but the committed table is available, the cells are compared: order-insensitive, and
floats within 1e-9 (a rebuild on the same machine already differs by float rounding of about 1e-15 in five tables,
see `validation/lineage`). PASS needs equal rows and equal bytes or equal content.

In [ ]:
import hashlib

import numpy as np
import pandas as pd


def sha256(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()


def content_diff(ref: Path, new: Path) -> str:
    """Compare two tables cell by cell, ignoring row order; floats compared within 1e-9."""
    a, b = pd.read_parquet(ref), pd.read_parquet(new)
    if list(a.columns) != list(b.columns) or len(a) != len(b):
        return "columns or rows differ"
    order = [c for c in a.columns if not pd.api.types.is_float_dtype(a[c])]
    try:
        a, b = (x.sort_values(order, kind="stable").reset_index(drop=True) if order else x for x in (a, b))
    except TypeError:  # a column of lists or dicts: compare in stored order
        pass
    bad = []
    for c in a.columns:
        if pd.api.types.is_float_dtype(a[c]):
            x, y = a[c].to_numpy(float), b[c].to_numpy(float)
            if not np.allclose(x, y, rtol=0, atol=1e-9, equal_nan=True):
                bad.append(f"{c} (max {np.nanmax(np.abs(x - y)):.1e})")
        elif not a[c].astype(str).equals(b[c].astype(str)):
            bad.append(c)
    return "content equal" if not bad else "differs: " + ", ".join(bad[:6])


REF = json.loads((REPO / "derived/v1/MANIFEST.json").read_text())
ref_hash = {name: REF["other_input_sha256"].get(src) for name, src in COPY.items()}
report = []
for name, t in REF["tables"].items():
    new, committed = DST / name, REPO / "derived/v1" / name
    if not new.exists():
        report.append((name, t["rows"], None, "not rebuilt", "FAIL"))
        continue
    rows = pq.ParquetFile(new).metadata.num_rows
    expected = ref_hash.get(name) or (sha256(committed) if committed.exists() else None)
    if expected is None:
        how = "no reference hash (rows only)"
    elif sha256(new) == expected:
        how = "sha256 equal"
    elif committed.exists():
        how = "sha256 differs; " + content_diff(committed, new)
    else:
        how = "sha256 differs"
    good = rows == t["rows"] and (how == "sha256 equal" or how.endswith("content equal") or how.startswith("no ref"))
    report.append((name, t["rows"], rows, how, "PASS" if good else "FAIL"))
for name, t in REF["csv_copies"].items():
    new = DST / name
    rows = sum(1 for _ in open(new, encoding="utf-8")) - 1 if new.exists() else None
    report.append((name, t["rows"], rows, "csv copy, rows only", "PASS" if rows == t["rows"] else "FAIL"))
check = pd.DataFrame(report, columns=["table", "manifest_rows", "rebuilt_rows", "comparison", "result"])
print(check.to_string(index=False))
print(f"{(check.result == 'PASS').sum()} of {len(check)} tables pass")